# Liu2024 — Faithful TWFB + DGFMDRM (FgMDM) Reproduction

A direct Python port of `TWFB_DGFMDM.m`. The goal is **diagnostic**: confirm whether this
project's data + labels can reproduce Liu2024's ~72% with their own classical pipeline. If yes,
the data is sound and the S-JEPA gap is a model/data-regime problem. If no, there is a
data/label/channel issue that is also hurting S-JEPA.

**Exactly what the MATLAB does, ported here:**
- Native **500 Hz**; per trial, find the trigger (marker channel `== 2`) and take the
  **0–4 s** window after it (2000 samples). Filtering is done on the full epoch, then cropped,
  so band-pass edge transients do not enter the covariance.
- **50 Hz notch** + band-pass over the **8 Liu bands**
  `{[8,12],[8,20],[8,30],[12,20],[15,20],[15,30],[20,30],[8,15]}`.
- **SCM covariance** `C = Xᵀ X` (unnormalized; scale is irrelevant to the affine-invariant metric).
- **DGFMDRM = `pyriemann.FgMDM`** — Fisher **geodesic discriminant filtering** (the "DGF" part)
  **+ MDM** (minimum distance to Riemannian mean), `metric='riemann'`. This is the Python
  equivalent of the toolbox `fgmdm` the MATLAB calls.
- Protocol: per subject, **10 random 24-train / 16-test splits**; the reported per-rep accuracy
  is the **max over the 8 bands** (matching the MATLAB's `max(acc_temp)`).

**Two honesty additions (clearly separated):**
- The MATLAB's max-over-bands picks the band on the **test** set → optimistic. We report that
  number (to match 72%) *and* an **unbiased nested** number where the band is chosen by an
  inner CV on the training split only.
- If `pyriemann` is missing we fall back to plain **log-Euclidean MDM (no DGF)** and label it as
  such — that is *not* the faithful FgMDM.

# 1. Setup

In [1]:
import sys
import os, re, json, hashlib, random, builtins, platform
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from scipy.io import loadmat
from scipy import signal
from scipy.linalg import eigh

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix

try:
    import matplotlib.pyplot as plt
    HAVE_MPL = True
except Exception as exc:
    HAVE_MPL = False
    print(f"[setup] matplotlib unavailable -> plots skipped: {exc}")

try:
    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset, Subset
    HAVE_TORCH = True
except Exception as exc:
    HAVE_TORCH = False
    Dataset = object
    print(f"[setup] torch unavailable -> EEGNet path skipped: {exc}")


try:
    from pyriemann.estimation import Covariances
    from pyriemann.classification import FgMDM, MDM
    HAVE_PYRIEMANN = True
except Exception as exc:
    HAVE_PYRIEMANN = False
    print(f"[setup] pyriemann unavailable -> FALLBACK to log-Euclidean MDM (NOT faithful FgMDM): {exc}")

print("MDM backend:",
      "pyriemann.FgMDM (faithful DGFMDRM)" if HAVE_PYRIEMANN else "log-Euclidean MDM fallback (no DGF)")


MDM backend: pyriemann.FgMDM (faithful DGFMDRM)


# 2. Configuration

## 2.1 Channel defaults (carried verbatim)

In [2]:
# Liu2024 source MAT channel conventions.
# Source files are organized as trials x 33 channels x samples:
#   0..29 = EEG-like channels, index 17 = CPz source reference,
#   30..31 = EOG, 32 = marker.
#
# The channel labels below follow the Liu2024 paper / EEGLAB location files.
# This matters for montage-dependent topomaps and any channel-position metadata.
SOURCE_EEG_CHANNEL_INDICES_30 = list(range(30))
SOURCE_REFERENCE_INDEX = 17
SOURCE_EEG_CHANNEL_INDICES_29 = [i for i in SOURCE_EEG_CHANNEL_INDICES_30 if i != SOURCE_REFERENCE_INDEX]

SOURCE_EEG_CHANNEL_NAMES_30 = [
    "Fp1", "Fp2", "Fz", "F3", "F4", "F7", "F8", "FCz", "FC3", "FC4",
    "FT7", "FT8", "Cz", "C3", "C4", "T3", "T4", "CPz",
    "CP3", "CP4", "TP7", "TP8", "Pz", "P3", "P4", "T5", "T6", "Oz", "O1", "O2",
]

# EOG and marker channels available in Liu2024 source MAT files.
SOURCE_EOG_CHANNEL_INDICES = [30, 31]
SOURCE_MARKER_CHANNEL_INDEX = 32


## 2.2 CONFIG

Reference CONFIG carried verbatim for paths/identity/constants, then a dedicated `liu` block
that encodes the MATLAB's choices. Note these intentionally **differ from the S-JEPA pipeline**
(native 500 Hz, 0–4 s from trigger, SCM covariance) — this is a faithful classical reproduction,
not a comparability run.

In [3]:
WORKING_DIR = Path.cwd().resolve().parent.parent

CONFIG = {
    # ------------------------------------------------------------------
    # Paths / run identity
    # ------------------------------------------------------------------
    "artifact_dir": str(WORKING_DIR / "artifacts" / "liu2024-source-mat-sjepa-prelocal"),
    "source_extract_dir": str(WORKING_DIR / "liu2024_data" / "liu2024_figshare" / "sourcedata"),
    "experiment_name": "baseline_sjepa_prelocal",
    "config_note": "Clean MNE-style preprocessing pipeline builder + S-JEPA hyperparameter controls.",

    # ------------------------------------------------------------------
    # Dataset
    # ------------------------------------------------------------------
    "subjects_to_use": None,
    "exclude_subjects": [],
    "source_unit": "microvolts",
    "final_model_unit": "microvolts",

    # ------------------------------------------------------------------
    # Source-domain preprocessing before creating MNE RawArray
    # ------------------------------------------------------------------
    "demean_mode": "none",      # none, trial_mean, baseline_window_mean
    "baseline_window_s": [0.0, 2.0],
    "detrend_mode": "none",                     # none, constant, linear
    "eog_correction": "none",                   # none, linear_regression

    # Robust source-domain clipping / winsorization. Use cautiously.
    "artifact_clip_mode": "none",               # none, absolute, percentile
    "artifact_clip_abs_value": None,             # in source_unit, e.g. 150.0 when source_unit=microvolts
    "artifact_clip_percentile": 99.5,

    # ------------------------------------------------------------------
    # MNE Raw-level preprocessing
    # ------------------------------------------------------------------
    "reference_mode": "average",                # average, none
    "reference_timing": "before_resample_filter",  # before_resample_filter, after_resample_before_filter, after_filter
    "resample": True,
    "resample_sfreq": 128,

    "filter_enabled": True,
    "filter_low": 0.5,
    "filter_high": 40.0,
    "filter_method": "fir",                     # fir, iir
    "filter_phase": "zero",                     # zero, zero-double, minimum (FIR only)
    "filter_fir_design": "firwin",              # firwin, firwin2 (FIR only)
    "filter_l_trans_bandwidth": "auto",         # auto or float, FIR only
    "filter_h_trans_bandwidth": "auto",         # auto or float, FIR only
    "filter_iir_params": None,                    # example: {"order": 2, "ftype": "butter"}

    "notch_freqs": None,                          # example: [50.0]
    "notch_before_bandpass": False,

    # ------------------------------------------------------------------
    # Windowing and post-window cleaning
    # ------------------------------------------------------------------
    "target_window_s": 4.2,
    "target_window_samples": 537,
    "mi_window_start_s": 1.5,

    "reject_bad_trials": False,
    "reject_peak_to_peak_threshold": None,       # in final_model_unit
    "reject_abs_threshold": None,                # in final_model_unit
    "min_trials_per_class_after_reject": None,

    # ------------------------------------------------------------------
    # Fold-safe normalization. train_* modes are fit on each training split only.
    # ------------------------------------------------------------------
    "normalization_mode": "none",               # none, train_global_zscore, train_channel_zscore, train_channel_robust, trial_global_zscore, trial_channel_zscore
    "normalization_eps": 1e-6,

    # ------------------------------------------------------------------
    # Model / downstream strategy
    # ------------------------------------------------------------------
    "model_name": "SignalJEPA_PreLocal",
    "pretrained_mode": "from_pretrained",
    "pretrained_repo_id": "braindecode/signal-jepa_without-chans",
    "strategy": "new",                          # new, full
    "warmup_epochs": 10,

    # ------------------------------------------------------------------
    # Evaluation protocol
    # ------------------------------------------------------------------
    "evaluation_mode": "stratified_kfold",      # stratified_kfold, liu2024_repeated_60_40, repeated_stratified_split
    "cv_folds": 5,
    "n_repeats": 10,
    "test_size": 0.4,
    "split_random_state": 2026,
    "assert_balanced_folds": True,

    # ------------------------------------------------------------------
    # Training hyperparameters
    # ------------------------------------------------------------------
    "batch_size": 4,
    "n_epochs": 5000,
    "early_stopping_patience": 50,
    "val_split": 0.2,
    "learning_rate": 0.0003,
    "optimizer_name": "adam",                   # adam, adamw
    "weight_decay": 0.0,
    "gradient_clip_norm": None,
    "checkpoint_metric": "valid_loss",           # valid_loss, valid_balanced_accuracy
    "label_smoothing": 0.0,
    "prediction_balance_loss_weight": 1.0,

    # ------------------------------------------------------------------
    # braindecode on-the-fly augmentation.
    # Applied to the TRAINING iterator ONLY (via AugmentedDataLoader), so the
    # validation split skorch carves out internally is never augmented -> no leakage.
    # There is no fixed "number of augmented samples": the model sees a freshly
    # augmented view of the train fold every epoch. Control INTENSITY with each
    # transform's "probability" (how often it fires) and its magnitude params.
    # Ready-to-use configs are in the markdown cell just below CONFIG.
    # ------------------------------------------------------------------
    # "augmentation": {
    #     "enabled": False,        # master switch
    #     "name": "none",          # label, saved with artifacts
    #     "random_state": 2026,
    #     # each entry: {"name": <transform>, "probability": 0..1, <transform params>}
    #     "transforms": [],
    # },

    "augmentation": {
        "enabled": True,
        "name": "time_mask",
        "random_state": 2026,
        "transforms": [
        {
            "mask_len_samples": 64,
            "name": "smooth_time_mask",
            "probability": 0.5
        }
        ]
    },

    # ------------------------------------------------------------------
    # Reproducibility
    # ------------------------------------------------------------------
    "seed": 2026,
    "set_seed": True,
    "cv_random_state": 2026,
    "val_split_random_state": 2026,

    # ------------------------------------------------------------------
    # Diagnostics / interpretation
    # ------------------------------------------------------------------
    "extract_spatial_conv_weights": True,
    "save_spatial_weight_plots": False,
    "plot_individual_spatial_filters": False,
    "max_spatial_filters_to_plot": 8,
    "topomap_dpi": 160,
    "topomap_value_mode": "relative_zscore",    # raw, relative_zscore, relative_percent
    "topomap_cmap": "RdBu_r",
    "collapse_threshold": 0.9,
    "log_spatial_update_stats": True,
    "log_probability_diagnostics": True,
}

# ------------------------------------------------------------------ #
#  Faithful Liu2024 TWFB+DGFMDRM reproduction settings.              #
# ------------------------------------------------------------------ #
CONFIG["experiment_name"] = "twfb_dgfmdm_faithful_reproduction"
CONFIG["artifact_dir"] = str(WORKING_DIR / "artifacts" / "liu2024-twfb-dgfmdm-faithful")
CONFIG["config_note"] = "Direct port of TWFB_DGFMDM.m: native 500 Hz, 0-4 s from trigger, SCM cov, FgMDM, 8 bands, 24/16 x10."

CONFIG["liu"] = {
    # --- channels: MATLAB channel = [1:17 19:30] (1-based) -> drop the reference (idx 17, 0-based) ---
    "keep_channel_indices": list(range(0, 17)) + list(range(18, 30)),   # 29 EEG channels, 0-based
    "marker_channel_index": 32,           # MATLAB column 33
    "trigger_value": 2,                   # MATLAB find(col33==2)
    "sfreq_hz": 500,                      # native; no resample
    # --- window: 0..4 s after trigger (MATLAB keeps samples 801:2800 of the trigger-aligned grab) ---
    "window_start_after_trigger_s": 0.0,
    "window_length_s": 4.0,
    "fallback_cue_sample": 750,           # used only if a trial has no detectable trigger (1.5 s * 500)
    # --- filtering ---
    "notch_hz": 50.0,
    "filter_order": 4,
    "freq_bands_hz": [[8, 12], [8, 20], [8, 30], [12, 20], [15, 20], [15, 30], [20, 30], [8, 15]],
    # --- classifier / metric ---
    "metric": "riemann",
    "cov_estimator": "scm",               # SCM == X^T X (matches MATLAB SS'*SS up to scale)
    "cov_shrinkage_fallback": 1e-3,       # only for the log-Euclidean fallback
    # --- protocol (MATLAB: 10 reps, 24 train / 16 test, max over bands) ---
    "n_repeats": 10,
    "n_train": 24,
    "n_test": 16,
    "stratified_split": False,            # MATLAB uses plain randperm; set True for a balanced variant
    "nested_inner_folds": 3,              # for the unbiased band-selection number
    "max_subjects": None,
    "base_split_seed": 2026,
}
print(f"Experiment: {CONFIG['experiment_name']}")
print(f"bands: {CONFIG['liu']['freq_bands_hz']}")
print(f"protocol: {CONFIG['liu']['n_repeats']}x {CONFIG['liu']['n_train']}/{CONFIG['liu']['n_test']} split")


Experiment: twfb_dgfmdm_faithful_reproduction
bands: [[8, 12], [8, 20], [8, 30], [12, 20], [15, 20], [15, 30], [20, 30], [8, 15]]
protocol: 10x 24/16 split


## 2.3 Derived constants / artifacts / reproducibility (carried verbatim)

In [4]:
# Liu2024 source MAT constants.
LIU_SOURCE_SFREQ = 500
LIU_EXPECTED_TRIALS_PER_SUBJECT = 40
LIU_EXPECTED_SOURCE_CHANNELS = 33
LIU_EXPECTED_SOURCE_SAMPLES_PER_TRIAL = 4000

# Liu source MAT files include EEG + EOG + marker channels.
# Keep the 29 EEG channels used in the Liu paper baseline and drop CPz because it is the source reference channel.
EEG_CHANNEL_INDICES = SOURCE_EEG_CHANNEL_INDICES_29
EEG_CHANNEL_NAMES = [
    name for idx, name in enumerate(SOURCE_EEG_CHANNEL_NAMES_30)
    if idx != SOURCE_REFERENCE_INDEX
]

SOURCE_EXTRACT_DIR = Path(CONFIG["source_extract_dir"])
TARGET_N_CLASSES = 2

if bool(CONFIG.get("resample", True)):
    EFFECTIVE_SFREQ = float(CONFIG.get("resample_sfreq", 128))
else:
    EFFECTIVE_SFREQ = float(LIU_SOURCE_SFREQ)

CONFIG["effective_sfreq"] = EFFECTIVE_SFREQ
CONFIG["sfreq"] = EFFECTIVE_SFREQ  # compatibility with existing cells/artifacts

if CONFIG.get("target_window_samples", None) is None:
    WINDOW_SAMPLES = int(round(float(CONFIG["target_window_s"]) * EFFECTIVE_SFREQ))
else:
    WINDOW_SAMPLES = int(CONFIG["target_window_samples"])

TARGET_TRIAL_DURATION_S = WINDOW_SAMPLES / EFFECTIVE_SFREQ
MI_WINDOW_START_SAMPLE = int(round(float(CONFIG["mi_window_start_s"]) * EFFECTIVE_SFREQ))
MI_WINDOW_STOP_SAMPLE = MI_WINDOW_START_SAMPLE + WINDOW_SAMPLES

PREPROCESSING_KEYS = [
    "source_unit", "final_model_unit",
    "demean_mode", "baseline_window_s", "detrend_mode", "eog_correction",
    "artifact_clip_mode", "artifact_clip_abs_value", "artifact_clip_percentile",
    "reference_mode", "reference_timing", "resample", "resample_sfreq", "effective_sfreq",
    "filter_enabled", "filter_low", "filter_high", "filter_method", "filter_phase",
    "filter_fir_design", "filter_l_trans_bandwidth", "filter_h_trans_bandwidth", "filter_iir_params",
    "notch_freqs", "notch_before_bandpass",
    "mi_window_start_s", "target_window_s", "target_window_samples",
    "reject_bad_trials", "reject_peak_to_peak_threshold", "reject_abs_threshold",
    "normalization_mode", "normalization_eps",
]

TRAINING_KEYS = [
    "strategy", "batch_size", "learning_rate", "optimizer_name", "weight_decay",
    "val_split", "early_stopping_patience", "n_epochs",
    "augmentation",
]

EVALUATION_KEYS = [
    "evaluation_mode", "cv_folds", "n_repeats", "test_size",
    "cv_random_state", "split_random_state", "val_split_random_state",
]

def summarize_selected_config(keys):
    return {k: CONFIG.get(k) for k in keys}

PREPROCESSING_CONFIG = summarize_selected_config(PREPROCESSING_KEYS)
TRAINING_CONFIG = summarize_selected_config(TRAINING_KEYS)
EVALUATION_CONFIG = summarize_selected_config(EVALUATION_KEYS)

def print_config_block(title, values):
    print(title)
    for key, value in values.items():
        print(f"  {key:34s}: {value}")

print("Effective Liu2024 Source MAT settings:")
print(f"  Experiment:                        {CONFIG.get('experiment_name')}")
print(f"  Note:                              {CONFIG.get('config_note')}")
print(f"  Channels:                          {len(EEG_CHANNEL_NAMES)}")
print(f"  Channel names:                     {EEG_CHANNEL_NAMES}")
print(f"  Source sfreq:                      {LIU_SOURCE_SFREQ} Hz")
print(f"  Effective sfreq:                   {EFFECTIVE_SFREQ} Hz")
print(f"  MI window start / samples:         {CONFIG['mi_window_start_s']} s / {WINDOW_SAMPLES}")
print(f"  Effective window duration:         {TARGET_TRIAL_DURATION_S:.4f} s")
print(f"  Evaluation mode:                   {CONFIG.get('evaluation_mode')}")
print(f"  Fixed seed:                        base={CONFIG.get('seed')} | cv={CONFIG.get('cv_random_state')} | split={CONFIG.get('split_random_state')} | val={CONFIG.get('val_split_random_state')}")
print_config_block("\nPreprocessing config:", PREPROCESSING_CONFIG)
print_config_block("\nTraining config:", TRAINING_CONFIG)
print_config_block("\nEvaluation config:", EVALUATION_CONFIG)


Effective Liu2024 Source MAT settings:
  Experiment:                        twfb_dgfmdm_faithful_reproduction
  Note:                              Direct port of TWFB_DGFMDM.m: native 500 Hz, 0-4 s from trigger, SCM cov, FgMDM, 8 bands, 24/16 x10.
  Channels:                          29
  Channel names:                     ['Fp1', 'Fp2', 'Fz', 'F3', 'F4', 'F7', 'F8', 'FCz', 'FC3', 'FC4', 'FT7', 'FT8', 'Cz', 'C3', 'C4', 'T3', 'T4', 'CP3', 'CP4', 'TP7', 'TP8', 'Pz', 'P3', 'P4', 'T5', 'T6', 'Oz', 'O1', 'O2']
  Source sfreq:                      500 Hz
  Effective sfreq:                   128.0 Hz
  MI window start / samples:         1.5 s / 537
  Effective window duration:         4.1953 s
  Evaluation mode:                   stratified_kfold
  Fixed seed:                        base=2026 | cv=2026 | split=2026 | val=2026

Preprocessing config:
  source_unit                       : microvolts
  final_model_unit                  : microvolts
  demean_mode                       : none
  bas

In [5]:
def create_run_id():
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    config_str = json.dumps(CONFIG, sort_keys=True, default=str)
    config_hash = hashlib.md5(config_str.encode()).hexdigest()[:8]
    return f"{timestamp}_{config_hash}"

RUN_ID = create_run_id()
ARTIFACT_DIR = Path(CONFIG["artifact_dir"]) / RUN_ID
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

LOG_PATH = ARTIFACT_DIR / "run.log"
_LOG_FILE_HANDLE = open(LOG_PATH, "a", buffering=1, encoding="utf-8", errors="replace")

def _safe_write_text(stream, text):
    try:
        stream.write(text)
        return
    except UnicodeEncodeError:
        pass

    encoding = getattr(stream, "encoding", None) or "utf-8"
    safe_text = text.encode(encoding, errors="replace").decode(encoding, errors="replace")
    stream.write(safe_text)

def _timestamped_print(*args, **kwargs):
    sep = kwargs.pop("sep", " ")
    end = kwargs.pop("end", "\n")
    flush = kwargs.pop("flush", False)
    file = kwargs.pop("file", None)
    message = sep.join(str(arg) for arg in args)
    leading_newlines = len(message) - len(message.lstrip("\n"))
    message_body = message[leading_newlines:]

    def _write_target(text):
        if file is None:
            _safe_write_text(sys.stdout, text)
            if flush:
                sys.stdout.flush()
        else:
            _safe_write_text(file, text)
            if flush and hasattr(file, "flush"):
                file.flush()

    if leading_newlines > 0:
        blanks = "\n" * leading_newlines
        _write_target(blanks)
        _safe_write_text(_LOG_FILE_HANDLE, blanks)

    if message_body:
        ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        stamped = f"[{ts}] {message_body}"
        _write_target(stamped + end)
        _safe_write_text(_LOG_FILE_HANDLE, stamped + end)
    else:
        _write_target(end)
        _safe_write_text(_LOG_FILE_HANDLE, end)

    if flush:
        _LOG_FILE_HANDLE.flush()

builtins.print = _timestamped_print

config_path = ARTIFACT_DIR / "config.json"
with open(config_path, "w") as f:
    json.dump(CONFIG, f, indent=2)

print(f"Run ID:     {RUN_ID}")
print(f"Artifacts:  {ARTIFACT_DIR}")
print(f"Config:     {config_path}")


[2026-06-13 09:50:19] Run ID:     20260613_0950_cc386b17
[2026-06-13 09:50:19] Artifacts:  /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/artifacts/liu2024-twfb-dgfmdm-faithful/20260613_0950_cc386b17
[2026-06-13 09:50:19] Config:     /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/artifacts/liu2024-twfb-dgfmdm-faithful/20260613_0950_cc386b17/config.json


In [6]:
def resolve_device():
    if torch.backends.mps.is_available() and torch.backends.mps.is_built():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")

DEVICE = resolve_device()
print(f"Using device: {DEVICE}")

def seed_everything(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)

BASE_SEED = int(CONFIG["seed"])
if CONFIG["set_seed"]:
    seed_everything(BASE_SEED)
    print(f"Seed initialized: {BASE_SEED}")


[2026-06-13 09:50:19] Using device: mps
[2026-06-13 09:50:20] Seed initialized: 2026


## 2.4 Data-loading helpers (carried verbatim — used only to read the raw 500 Hz `.mat`)

In [7]:
def find_source_mat_files(root):
    root = Path(root)
    return sorted(root.rglob("*.mat")) if root.exists() else []

def subject_id_from_path(path):
    s = str(path)
    m = re.search(r"sub[-_ ]?(\d{1,2})", s, flags=re.IGNORECASE)
    if m:
        return int(m.group(1))
    nums = re.findall(r"\d+", Path(path).stem)
    if nums:
        return int(nums[-1])
    raise ValueError(f"Could not infer subject id from path: {path}")

def _is_mat_struct(x):
    return hasattr(x, "_fieldnames")

def _walk_mat_object(obj, prefix=""):
    """Recursively walk scipy-loaded MATLAB dicts/structs.

    Liu2024 source files may expose only a top-level `eeg` object instead of
    top-level `rawdata` and `labels`. This walker lets the loader find nested
    arrays without assuming one exact MATLAB struct layout.
    """
    if isinstance(obj, dict):
        for k, v in obj.items():
            if str(k).startswith("__"):
                continue
            name = f"{prefix}.{k}" if prefix else str(k)
            yield name, v
            yield from _walk_mat_object(v, name)
    elif _is_mat_struct(obj):
        for k in obj._fieldnames:
            v = getattr(obj, k)
            name = f"{prefix}.{k}" if prefix else str(k)
            yield name, v
            yield from _walk_mat_object(v, name)
    elif isinstance(obj, np.ndarray):
        if obj.dtype == object and obj.size == 1:
            yield from _walk_mat_object(obj.item(), prefix)
        elif obj.dtype == object:
            for idx, item in np.ndenumerate(obj):
                yield from _walk_mat_object(item, f"{prefix}{idx}")

def mat_structure_preview(path, max_rows=200):
    mat = loadmat(path, squeeze_me=True, struct_as_record=False)
    rows = []
    for name, value in _walk_mat_object(mat):
        if isinstance(value, np.ndarray):
            rows.append({
                "name": name,
                "type": "ndarray",
                "shape": str(value.shape),
                "dtype": str(value.dtype),
            })
        else:
            rows.append({
                "name": name,
                "type": type(value).__name__,
                "shape": "",
                "dtype": "",
            })
    return pd.DataFrame(rows).head(max_rows)

def _normalize_rawdata_shape(rawdata, labels=None):
    arr = np.asarray(rawdata)
    if arr.ndim != 3:
        raise ValueError(f"Expected 3D rawdata, got shape={arr.shape}")

    # Prefer the label-count axis as the trial axis when labels are available.
    if labels is not None:
        n_labels = int(np.asarray(labels).size)
        trial_axes = [axis for axis, size in enumerate(arr.shape) if size == n_labels]
    else:
        trial_axes = []

    if not trial_axes:
        trial_axes = [axis for axis, size in enumerate(arr.shape) if size in (39, 40)]

    if trial_axes and trial_axes[0] != 0:
        arr = np.moveaxis(arr, trial_axes[0], 0)

    # After trial-axis normalization, the time axis should be the largest axis.
    time_axis = int(np.argmax(arr.shape[1:]) + 1)
    if time_axis != 2:
        arr = np.moveaxis(arr, time_axis, 2)

    if arr.shape[1] < 29 or arr.shape[2] < 1000:
        raise ValueError(f"Could not normalize rawdata to trials x channels x samples, got {arr.shape}")
    return arr

def _score_raw_candidate(name, arr):
    lname = name.lower()
    score = 0
    if "rawdata" in lname or "raw" in lname or "data" in lname:
        score += 10
    if arr.ndim == 3:
        score += 5
    if any(size in (39, 40) for size in arr.shape):
        score += 3
    if max(arr.shape) >= 3000:
        score += 2
    if "eeg" in lname:
        score += 1
    return score

def _score_label_candidate(name, arr):
    lname = name.lower()
    flat = np.asarray(arr).ravel()
    unique = set(np.unique(flat).astype(str).tolist()) if flat.size <= 200 else set()
    score = 0
    if "label" in lname or "class" in lname or lname.split(".")[-1] in {"y", "labels"}:
        score += 10
    if flat.size in (39, 40):
        score += 3
    if unique and unique.issubset({"0", "1", "2"}):
        score += 2
    return score

def validate_liu_source_subject(rawdata, labels, subject_id, path=None):
    """Validate the fixed Liu source MAT layout assumptions."""
    expected_trials = LIU_EXPECTED_TRIALS_PER_SUBJECT
    expected_channels = LIU_EXPECTED_SOURCE_CHANNELS
    expected_samples = LIU_EXPECTED_SOURCE_SAMPLES_PER_TRIAL

    if rawdata.shape[0] != labels.size:
        raise ValueError(
            f"Subject {subject_id}: labels/trials mismatch. "
            f"rawdata={rawdata.shape}, labels={labels.shape}, path={path}"
        )
    if rawdata.shape[0] != expected_trials:
        print(f"WARNING subject {subject_id}: expected {expected_trials} trials, got {rawdata.shape[0]}")
    if rawdata.shape[1] < len(SOURCE_EEG_CHANNEL_INDICES_30):
        raise ValueError(f"Subject {subject_id}: expected at least 30 EEG-like channels, got {rawdata.shape}")
    if rawdata.shape[1] != expected_channels:
        print(f"WARNING subject {subject_id}: expected {expected_channels} source channels, got {rawdata.shape[1]}")
    if rawdata.shape[2] != expected_samples:
        print(f"WARNING subject {subject_id}: expected {expected_samples} samples/trial, got {rawdata.shape[2]}")

    unique = set(np.unique(labels).astype(int).tolist())
    if not unique.issubset({0, 1, 2}):
        raise ValueError(f"Subject {subject_id}: unexpected labels {sorted(unique)}")

    y0 = labels_to_zero_based(labels)
    counts = np.bincount(y0, minlength=TARGET_N_CLASSES)
    if counts.min() == 0:
        raise ValueError(f"Subject {subject_id}: missing class after zero-based conversion, counts={counts.tolist()}")
    if counts[0] != counts[1]:
        print(f"WARNING subject {subject_id}: class counts are not balanced: {counts.tolist()}")

def load_subject_mat(path):
    mat = loadmat(path, squeeze_me=True, struct_as_record=False)

    arrays = []
    for name, value in _walk_mat_object(mat):
        if isinstance(value, np.ndarray) and value.dtype != object:
            arrays.append((name, np.asarray(value)))

    raw_candidates, label_candidates = [], []
    for name, arr in arrays:
        if arr.ndim == 3:
            raw_candidates.append((_score_raw_candidate(name, arr), name, arr))
        elif arr.ndim in (1, 2):
            label_candidates.append((_score_label_candidate(name, arr), name, arr))

    if not raw_candidates or not label_candidates:
        preview = mat_structure_preview(path)
        preview_path = ARTIFACT_DIR / f"mat_structure_failure_{Path(path).stem}.csv"
        preview.to_csv(preview_path, index=False)
        print(f"MAT structure preview for failure saved to: {preview_path}")
        display(preview.head(40))
        raise KeyError(f"Could not locate 3D raw data and labels in {path}")

    _, raw_name, raw_arr = sorted(raw_candidates, key=lambda x: x[0], reverse=True)[0]
    _, label_name, label_arr = sorted(label_candidates, key=lambda x: x[0], reverse=True)[0]

    labels = np.asarray(label_arr).astype(int).ravel()
    rawdata = _normalize_rawdata_shape(raw_arr, labels=labels).astype(np.float64)

    if labels.size != rawdata.shape[0]:
        raise ValueError(f"Label count mismatch in {path}: labels={labels.shape}, rawdata={rawdata.shape}")

    return rawdata, labels.astype(int), raw_name, label_name


# 3. Liu-faithful windowing, filtering, and covariance

We read each subject's raw `(trials, 33, 4000)` array, detect the trigger per trial from the
marker channel, select the 29 EEG channels, notch + band-pass the full epoch, then crop the
0–4 s window and form the SCM covariance. Everything matches the MATLAB; where a trial lacks a
detectable trigger we fall back to a fixed cue sample and **report** how often that happens.

In [8]:
LIU = CONFIG["liu"]
FS = float(LIU["sfreq_hz"])
WIN_LEN = int(round(LIU["window_length_s"] * FS))                 # 2000 samples
WIN_START = int(round(LIU["window_start_after_trigger_s"] * FS))  # 0
KEEP = np.array(LIU["keep_channel_indices"], dtype=int)

def detect_trigger(trial_all_channels):
    """Return the 0-4 s window start sample for one trial (trials are 33 x 4000 here)."""
    midx = LIU["marker_channel_index"]
    if trial_all_channels.shape[0] > midx:
        marker = np.rint(trial_all_channels[midx]).astype(int)
        hits = np.where(marker == int(LIU["trigger_value"]))[0]
        if hits.size:
            return int(hits[0]) + WIN_START, True
    return int(LIU["fallback_cue_sample"]) + WIN_START, False

def _notch(x, fs, f0, q=30.0):
    b, a = signal.iirnotch(f0 / (fs / 2.0), q)
    return signal.filtfilt(b, a, x, axis=-1)

def _bandpass(x, lo, hi, fs, order):
    nyq = 0.5 * fs
    b, a = signal.butter(order, [max(lo / nyq, 1e-4), min(hi / nyq, 0.999)], btype="band")
    return signal.filtfilt(b, a, x, axis=-1)

def subject_band_windows(raw_trials):
    """raw_trials: (n_trials, 33, 4000). Returns dict band_label -> (n_trials, 29, WIN_LEN) and trigger info."""
    n_tr = raw_trials.shape[0]
    starts, detected = [], 0
    for i in range(n_tr):
        s, ok = detect_trigger(raw_trials[i])
        # clamp so the window fits inside the epoch
        s = min(max(s, 0), raw_trials.shape[2] - WIN_LEN)
        starts.append(s); detected += int(ok)
    out = {}
    for lo, hi in LIU["freq_bands_hz"]:
        blab = f"{lo}-{hi}"
        W = np.empty((n_tr, KEEP.size, WIN_LEN))
        for i in range(n_tr):
            x = raw_trials[i, KEEP, :]                       # (29, 4000)
            x = _notch(x, FS, LIU["notch_hz"])               # 50 Hz notch on full epoch
            x = _bandpass(x, lo, hi, FS, LIU["filter_order"])# band-pass on full epoch (clean edges)
            W[i] = x[:, starts[i]:starts[i] + WIN_LEN]       # crop 0-4 s
        out[blab] = W
    return out, detected, n_tr

def covariances(windows):
    """(n_trials, 29, T) -> (n_trials, 29, 29). pyriemann SCM if available else numpy SCM."""
    if HAVE_PYRIEMANN:
        return Covariances(estimator=LIU["cov_estimator"]).transform(windows)
    n_tr, n_ch, T = windows.shape
    C = np.empty((n_tr, n_ch, n_ch))
    for i in range(n_tr):
        Xi = windows[i]
        C[i] = Xi @ Xi.T                                      # SCM == X X^T (matches SS'*SS)
    return C

def labels_zero_based(labels):
    """MATLAB labels are 1 (left) / 2 (right). Map to 0/1; class 1 == right hand."""
    return np.asarray(labels, dtype=int) - 1


# 4. DGFMDRM classifier (`FgMDM`) and the fallback

`classify_fold` fits on the training covariances and predicts the test covariances. With
`pyriemann` it is the faithful **FgMDM** (geodesic discriminant filtering + MDM). Without it,
a log-Euclidean MDM (no DGF) is used and flagged.

In [9]:
def _logm_spd(C):
    w, V = eigh(C); w = np.clip(w, 1e-12, None)
    return (V * np.log(w)) @ V.T

def _logeuclid_mdm(cov_tr, y_tr, cov_te, n_classes, shrink):
    # regularize toward scaled identity, take matrix logs, classify by nearest mean-of-logs
    def reg(C):
        n = C.shape[0]; tr = np.trace(C) / n
        return (1 - shrink) * C + shrink * tr * np.eye(n)
    logs_tr = np.stack([_logm_spd(reg(C)) for C in cov_tr])
    logs_te = np.stack([_logm_spd(reg(C)) for C in cov_te])
    means = []
    for c in range(n_classes):
        m = (y_tr == c)
        means.append(logs_tr[m].mean(axis=0) if m.any() else np.zeros_like(logs_tr[0]))
    means = np.stack(means)
    d = np.stack([np.linalg.norm(logs_te - mu, axis=(1, 2)) for mu in means], axis=1)
    return d.argmin(axis=1).astype(int)

def classify_fold(cov_tr, y_tr, cov_te, n_classes=2):
    if HAVE_PYRIEMANN:
        clf = FgMDM(metric=LIU["metric"])
        clf.fit(cov_tr, y_tr)
        return np.asarray(clf.predict(cov_te), dtype=int)
    return _logeuclid_mdm(cov_tr, y_tr, cov_te, n_classes, LIU["cov_shrinkage_fallback"])


# 5. Load raw subjects (500 Hz)

Uses the carried loader to read each `.mat` as `(trials, 33, 4000)` at the source rate. No MNE
resample/filter pipeline is used here — preprocessing is the Liu-faithful path above.

In [10]:
SRC = Path(CONFIG["source_extract_dir"])
mat_files = find_source_mat_files(SRC)
print(f"Found {len(mat_files)} .mat files under {SRC}")
if not mat_files:
    raise FileNotFoundError(f"No .mat files under {SRC}. Set CONFIG['source_extract_dir'].")

RAW_SUBJECTS = {}
for path in mat_files:
    sid = subject_id_from_path(path)
    rawdata, labels, _, _ = load_subject_mat(path)     # (trials, 33, 4000), labels in {1,2}
    RAW_SUBJECTS[str(sid)] = (rawdata, labels_zero_based(labels))
SUBJECTS = sorted(RAW_SUBJECTS.keys(), key=lambda s: int(s))
if LIU["max_subjects"]:
    SUBJECTS = SUBJECTS[:int(LIU["max_subjects"])]
print(f"Loaded {len(SUBJECTS)} subjects. Example raw shape: {RAW_SUBJECTS[SUBJECTS[0]][0].shape}")
print(f"Window: {WIN_LEN} samples = {LIU['window_length_s']}s @ {FS:.0f} Hz, starting {LIU['window_start_after_trigger_s']}s after trigger")


[2026-06-13 09:50:20] Found 50 .mat files under /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/liu2024_data/liu2024_figshare/sourcedata
[2026-06-13 09:50:27] Loaded 50 subjects. Example raw shape: (40, 33, 4000)
[2026-06-13 09:50:27] Window: 2000 samples = 4.0s @ 500 Hz, starting 0.0s after trigger


# 6. Precompute per-band covariances per subject

Covariances do not depend on the train/test split, so we compute them once per subject per band
(40 trials × 8 bands), then reuse across the 10 repetitions. Also records trigger-detection rate.

In [11]:
SUBJECT_COVS = {}     # sid -> {band_label: (n_trials, 29, 29)}
SUBJECT_LABELS = {}   # sid -> y (0/1)
trigger_report = []
band_labels = [f"{lo}-{hi}" for lo, hi in LIU["freq_bands_hz"]]

for sid in SUBJECTS:
    raw, y = RAW_SUBJECTS[sid]
    windows, detected, n_tr = subject_band_windows(raw)
    SUBJECT_COVS[sid] = {b: covariances(W) for b, W in windows.items()}
    SUBJECT_LABELS[sid] = y
    trigger_report.append({"subject_id": sid, "n_trials": n_tr,
                           "trigger_detected": detected, "fallback_used": n_tr - detected,
                           "class_counts": np.bincount(y, minlength=TARGET_N_CLASSES).tolist()})
trig_df = pd.DataFrame(trigger_report)
print(f"Trigger detection: {trig_df['trigger_detected'].sum()}/{trig_df['n_trials'].sum()} trials "
      f"({trig_df['fallback_used'].sum()} used the fixed-cue fallback).")
if trig_df["fallback_used"].sum() > 0:
    print("  NOTE: some trials had no marker==2; verify the marker channel / trigger value in CONFIG['liu'].")
trig_df.to_csv(ARTIFACT_DIR / "trigger_detection_report.csv", index=False)


[2026-06-13 09:51:43] Trigger detection: 2000/2000 trials (0 used the fixed-cue fallback).


# 7. Run the Liu protocol (10× 24/16 split, max-over-bands) + unbiased nested variant

For each subject and repetition: a random 24/16 split. For every band, fit FgMDM on train and
score test. We then record three things per rep:
- **per-band** test accuracy,
- **oracle max-over-bands** test accuracy (matches the MATLAB's reported number),
- **nested** accuracy: the band is chosen by an inner stratified CV on the **train** split only,
  then scored once on test (no test-set leakage in band selection).

In [12]:
def split_indices(y, n_train, n_test, rng, stratified):
    n = len(y)
    if stratified:
        order = []
        for c in np.unique(y):
            idx = np.where(y == c)[0]; rng.shuffle(idx); order.append(idx)
        # interleave to keep balance, then cut
        perm = np.concatenate(order); rng.shuffle(perm)
    else:
        perm = np.arange(n); rng.shuffle(perm)
    return perm[:n_train], perm[n_train:n_train + n_test]

def inner_best_band(covs_by_band, y_tr, tr_idx, n_folds):
    """Pick the band with the best inner-CV balanced accuracy on the train split only."""
    y = y_tr
    best_band, best_score = band_labels[0], -1.0
    # need at least 2 of each class for stratified folds
    folds = min(n_folds, int(np.min(np.bincount(y, minlength=TARGET_N_CLASSES))))
    if folds < 2:
        return band_labels[0]
    skf = StratifiedKFold(n_splits=folds, shuffle=True, random_state=int(LIU["base_split_seed"]))
    for b in band_labels:
        C = covs_by_band[b][tr_idx]
        scores = []
        for itr, ite in skf.split(np.arange(len(y)), y):
            try:
                yp = classify_fold(C[itr], y[itr], C[ite])
                scores.append(balanced_accuracy_score(y[ite], yp))
            except Exception:
                scores.append(0.0)
        s = float(np.mean(scores))
        if s > best_score:
            best_score, best_band = s, b
    return best_band

rows_band, rows_rep = [], []
for sid in SUBJECTS:
    covs_by_band = SUBJECT_COVS[sid]; y = SUBJECT_LABELS[sid]
    for rep in range(LIU["n_repeats"]):
        rng = np.random.default_rng(LIU["base_split_seed"] + 1000 * int(sid) + rep)
        tr_idx, te_idx = split_indices(y, LIU["n_train"], LIU["n_test"], rng, LIU["stratified_split"])
        y_tr, y_te = y[tr_idx], y[te_idx]
        band_acc = {}
        for b in band_labels:
            C = covs_by_band[b]
            yp = classify_fold(C[tr_idx], y_tr, C[te_idx])
            acc = accuracy_score(y_te, yp); bal = balanced_accuracy_score(y_te, yp)
            band_acc[b] = acc
            rows_band.append({"subject_id": sid, "rep": rep, "band": b,
                              "accuracy": acc, "balanced_accuracy": bal})
        # oracle: max over bands on test (matches MATLAB)
        oracle_band = max(band_acc, key=band_acc.get); oracle_acc = band_acc[oracle_band]
        # nested: choose band on train inner-CV, score once on test
        nb = inner_best_band(covs_by_band, y_tr, tr_idx, LIU["nested_inner_folds"])
        yp_nb = classify_fold(covs_by_band[nb][tr_idx], y_tr, covs_by_band[nb][te_idx])
        nested_acc = accuracy_score(y_te, yp_nb)
        rows_rep.append({"subject_id": sid, "rep": rep,
                         "oracle_maxband_accuracy": oracle_acc, "oracle_band": oracle_band,
                         "nested_accuracy": nested_acc, "nested_band": nb})
    print(f"  subject {sid}: oracle={np.mean([r['oracle_maxband_accuracy'] for r in rows_rep if r['subject_id']==sid]):.3f} "
          f"nested={np.mean([r['nested_accuracy'] for r in rows_rep if r['subject_id']==sid]):.3f}")

BAND_DF = pd.DataFrame(rows_band); REP_DF = pd.DataFrame(rows_rep)
BAND_DF.to_csv(ARTIFACT_DIR / "per_band_per_rep.csv", index=False)
REP_DF.to_csv(ARTIFACT_DIR / "per_rep_oracle_vs_nested.csv", index=False)


[2026-06-13 09:51:58]   subject 1: oracle=0.575 nested=0.425
[2026-06-13 09:52:12]   subject 2: oracle=0.619 nested=0.463
[2026-06-13 09:52:27]   subject 3: oracle=0.531 nested=0.412
[2026-06-13 09:52:40]   subject 4: oracle=0.556 nested=0.444
[2026-06-13 09:52:54]   subject 5: oracle=0.625 nested=0.438
[2026-06-13 09:53:10]   subject 6: oracle=0.588 nested=0.438
[2026-06-13 09:53:24]   subject 7: oracle=0.775 nested=0.688
[2026-06-13 09:53:38]   subject 8: oracle=0.562 nested=0.388
[2026-06-13 09:53:52]   subject 9: oracle=0.544 nested=0.425
[2026-06-13 09:54:06]   subject 10: oracle=0.562 nested=0.456
[2026-06-13 09:54:21]   subject 11: oracle=0.625 nested=0.481
[2026-06-13 09:54:36]   subject 12: oracle=0.637 nested=0.519
[2026-06-13 09:54:50]   subject 13: oracle=0.631 nested=0.487
[2026-06-13 09:55:05]   subject 14: oracle=0.537 nested=0.350
[2026-06-13 09:55:19]   subject 15: oracle=0.662 nested=0.531
[2026-06-13 09:55:34]   subject 16: oracle=0.525 nested=0.381
[2026-06-13 09:55

# 8. Headline numbers and comparison

In [13]:
# subject-level means, then group means (MATLAB reports % accuracy)
subj_oracle = REP_DF.groupby("subject_id")["oracle_maxband_accuracy"].mean()
subj_nested = REP_DF.groupby("subject_id")["nested_accuracy"].mean()
per_band_mean = BAND_DF.groupby("band")["accuracy"].mean().reindex(band_labels)

print("================ HEADLINE (accuracy, like the MATLAB) ================")
print(f"Oracle max-over-bands (matches Liu's reported metric): "
      f"{100*subj_oracle.mean():.1f}%  (subject sd {100*subj_oracle.std():.1f})")
print(f"Unbiased nested band-selection:                       "
      f"{100*subj_nested.mean():.1f}%  (subject sd {100*subj_nested.std():.1f})")
print(f"Best single fixed band ({per_band_mean.idxmax()}):                 "
      f"{100*per_band_mean.max():.1f}%")
print("\nMean accuracy per fixed band:")
for b in band_labels:
    print(f"  {b:>6} Hz : {100*per_band_mean[b]:.1f}%")
if not HAVE_PYRIEMANN:
    print("\n[!] pyriemann missing -> these used the log-Euclidean MDM FALLBACK (no DGF); "
          "install pyriemann for the faithful FgMDM reproduction.")

summary = {
    "backend": "pyriemann.FgMDM" if HAVE_PYRIEMANN else "logeuclid_mdm_fallback_no_dgf",
    "oracle_maxband_mean_accuracy": float(subj_oracle.mean()),
    "oracle_maxband_subject_std": float(subj_oracle.std()),
    "nested_mean_accuracy": float(subj_nested.mean()),
    "nested_subject_std": float(subj_nested.std()),
    "best_fixed_band": str(per_band_mean.idxmax()),
    "best_fixed_band_accuracy": float(per_band_mean.max()),
    "per_band_mean_accuracy": {b: float(per_band_mean[b]) for b in band_labels},
    "n_subjects": int(len(SUBJECTS)), "protocol": "10x 24/16 random split",
}
with open(ARTIFACT_DIR / "headline_summary.json", "w") as f:
    json.dump(summary, f, indent=2, default=str)

pd.DataFrame({"subject_id": subj_oracle.index,
              "oracle_maxband_accuracy": subj_oracle.values,
              "nested_accuracy": subj_nested.reindex(subj_oracle.index).values}
             ).to_csv(ARTIFACT_DIR / "per_subject_accuracy.csv", index=False)


[2026-06-13 10:04:04] ================ HEADLINE (accuracy, like the MATLAB) ================
[2026-06-13 10:04:04] Oracle max-over-bands (matches Liu's reported metric): 63.0%  (subject sd 8.3)
[2026-06-13 10:04:04] Unbiased nested band-selection:                       48.6%  (subject sd 9.5)
[2026-06-13 10:04:04] Best single fixed band (8-20):                 49.8%

[2026-06-13 10:04:04] Mean accuracy per fixed band:
[2026-06-13 10:04:04]     8-12 Hz : 47.7%
[2026-06-13 10:04:04]     8-20 Hz : 49.8%
[2026-06-13 10:04:04]     8-30 Hz : 49.8%
[2026-06-13 10:04:04]    12-20 Hz : 47.9%
[2026-06-13 10:04:04]    15-20 Hz : 47.1%
[2026-06-13 10:04:04]    15-30 Hz : 49.1%
[2026-06-13 10:04:04]    20-30 Hz : 49.0%
[2026-06-13 10:04:04]     8-15 Hz : 47.9%


# 9. Plots: per-band accuracy and per-subject spread

In [14]:
if HAVE_MPL and not BAND_DF.empty:
    # (a) accuracy by band (box across subject-reps)
    fig, ax = plt.subplots(figsize=(8, 4))
    data = [BAND_DF[BAND_DF["band"] == b]["accuracy"].values for b in band_labels]
    ax.boxplot(data, labels=band_labels, showmeans=True)
    ax.axhline(0.5, ls="--", c="grey", lw=1, label="chance")
    ax.set_ylabel("test accuracy"); ax.set_xlabel("filter band (Hz)")
    ax.set_title("FgMDM accuracy by band" if HAVE_PYRIEMANN else "MDM (fallback) accuracy by band")
    ax.legend(); fig.tight_layout()
    fig.savefig(ARTIFACT_DIR / "accuracy_by_band.png", dpi=160); plt.close(fig)
    print(f"Saved: {ARTIFACT_DIR/'accuracy_by_band.png'}")

    # (b) per-subject oracle vs nested
    s_or = REP_DF.groupby("subject_id")["oracle_maxband_accuracy"].mean()
    s_ne = REP_DF.groupby("subject_id")["nested_accuracy"].mean().reindex(s_or.index)
    order = s_or.sort_values().index
    x = np.arange(len(order))
    fig, ax = plt.subplots(figsize=(max(8, 0.3 * len(order)), 4))
    ax.bar(x - 0.2, s_or.reindex(order).values, width=0.4, label="oracle max-band")
    ax.bar(x + 0.2, s_ne.reindex(order).values, width=0.4, label="nested (unbiased)")
    ax.axhline(0.5, ls="--", c="grey", lw=1)
    ax.set_xticks(x); ax.set_xticklabels(order, rotation=90, fontsize=6)
    ax.set_ylabel("accuracy"); ax.set_ylim(0, 1)
    ax.set_title("Per-subject accuracy: oracle vs unbiased nested selection")
    ax.legend(); fig.tight_layout()
    fig.savefig(ARTIFACT_DIR / "per_subject_oracle_vs_nested.png", dpi=160); plt.close(fig)
    print(f"Saved: {ARTIFACT_DIR/'per_subject_oracle_vs_nested.png'}")
else:
    print("Plots skipped (need matplotlib + results).")


/var/folders/7d/njk_0cn503z09dk98r0csptr0000gn/T/ipykernel_35455/3619092830.py:5: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data, labels=band_labels, showmeans=True)


[2026-06-13 10:04:05] Saved: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/artifacts/liu2024-twfb-dgfmdm-faithful/20260613_0950_cc386b17/accuracy_by_band.png
[2026-06-13 10:04:05] Saved: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/artifacts/liu2024-twfb-dgfmdm-faithful/20260613_0950_cc386b17/per_subject_oracle_vs_nested.png


## 10. Save run metadata

In [15]:
run_metadata = {
    "run_id": RUN_ID, "artifact_dir": str(ARTIFACT_DIR),
    "experiment_name": CONFIG["experiment_name"], "config_note": CONFIG["config_note"],
    "backend": "pyriemann.FgMDM" if HAVE_PYRIEMANN else "logeuclid_mdm_fallback_no_dgf",
    "liu_config": LIU, "n_subjects": len(SUBJECTS), "headline": summary,
    "source": "faithful port of TWFB_DGFMDM.m",
}
with open(ARTIFACT_DIR / "run_metadata.json", "w") as f:
    json.dump(run_metadata, f, indent=2, default=str)
print(f"Saved run metadata to: {ARTIFACT_DIR/'run_metadata.json'}")
for p in sorted(ARTIFACT_DIR.glob("*")):
    print(f"  - {p.name}")
try:
    _LOG_FILE_HANDLE.close()
except Exception:
    pass


[2026-06-13 10:04:05] Saved run metadata to: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/artifacts/liu2024-twfb-dgfmdm-faithful/20260613_0950_cc386b17/run_metadata.json
[2026-06-13 10:04:05]   - accuracy_by_band.png
[2026-06-13 10:04:05]   - config.json
[2026-06-13 10:04:05]   - headline_summary.json
[2026-06-13 10:04:05]   - per_band_per_rep.csv
[2026-06-13 10:04:05]   - per_rep_oracle_vs_nested.csv
[2026-06-13 10:04:05]   - per_subject_accuracy.csv
[2026-06-13 10:04:05]   - per_subject_oracle_vs_nested.png
[2026-06-13 10:04:05]   - run.log
[2026-06-13 10:04:05]   - run_metadata.json
[2026-06-13 10:04:05]   - trigger_detection_report.csv


# 11. How to read this

- **If the oracle max-over-bands number lands near ~72%**, your data, labels, channel order, and
  windowing are sound — the S-JEPA gap is a model/data-regime problem, not a data bug. Use the
  per-band table to see which band carries the signal and feed that to S-JEPA.
- **The nested number is the honest one.** Expect it to be a few points below the oracle number;
  that gap *is* the optimism in the MATLAB's report-the-best-band protocol. Quote the nested
  number when comparing fairly against S-JEPA.
- **If even the oracle number is near chance**, stop and debug the data path before touching any
  model: check the trigger-detection report (Section 6), the {1,2}→{0,1} label mapping (class 1 =
  right hand), and the channel selection. A classical Riemannian pipeline at chance almost always
  means the inputs are wrong, and the same bug would sink S-JEPA.
- **Per-subject spread** (Section 9b) shows who is decodable; line this up with the
  consistently-hard subjects from the supervised-diagnostics notebook.

**Faithful vs. not:** with `pyriemann` this is FgMDM (DGF + MDM) on native-500 0–4 s SCM
covariances over the exact 8 bands and the 24/16×10 protocol — the closest practical match to
`TWFB_DGFMDM.m`. The only deliberate addition is the unbiased nested number. Differences that
remain: the MATLAB indexes a continuous recording and pads ±800 samples around the trigger,
whereas here epochs arrive pre-cut and we filter the full epoch before cropping (cleaner edges,
same intent); and exact filter implementations differ (scipy Butterworth vs. the MATLAB
`BandpassFilter`/`NotchFilter`).